# Containerizing ML Workflows with Docker
## Objectives
- Understand how to use Docker for ML workflows.
- Build a training container to encapsulate the model training pipeline.
- Build a serving container to expose the trained model as an API.
- Test both containers locally to ensure reproducibility.

## 1. Why Docker for ML?
**Problem:**
- You train a model on your system → works fine.
- Deploy to server/cloud → “ModuleNotFoundError” or “version mismatch” issues.

**Solution:**
- Docker bundles **code + dependencies + configs** into a container image.
- That image runs the same everywhere → laptop, cloud, Kubernetes, etc.

**Use Case Here:**
- **Training container** → runs **train.py** and **saves iris_model.pkl**.
- **Serving container** → loads **iris_model.pkl** and exposes **/predict** API.

## 2. Training Container Setup
We create a Dockerfile that installs Python, required libraries, and runs the training script.

### Dockerfile (training)

```dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt  # Include scikit-learn, mlflow, boto3
COPY train.py .
CMD ["python", "train.py"]

**Explanation:**

- **FROM python:3.9-slim** → Base image: lightweight Python 3.9.
- **WORKDIR /app** → Set working directory inside container.
- **COPY requirements.txt .** → Copy dependency list into container.
- **RUN pip install -r requirements.txt** → Install ML dependencies inside.
- **COPY train.py .** → Copy training script into container.
- **CMD ["python", "train.py"]** → Run training script by default when container starts.

### Build image

```bash
docker build -t iris-training:latest .

- Creates an docker image named **iris-training**.

### Run Container

```bash
docker start iris-training:latest

- The container executes **train.py**, trains the model, and saves it (e.g., **iris_model.pkl**).

## 3. Serving Container Setup

The serving container exposes the trained model as an API using **Flask**.

### Dockerfile (Serving)

```dockerfile
FROM python:3.9-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt  # Include scikit-learn, flask
COPY serve.py .
CMD ["python", "serve.py"]

**Explanation:**

- Same base image as training.
- Installs Flask (for API) + scikit-learn.
- Copies **serve.py** which runs the API.
- **Default command** → start Flask app.

### Sample serve.py

In [ ]:
from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)
model = joblib.load("iris_model.pkl")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json["features"]
    prediction = model.predict([data])
    return jsonify({"prediction": int(prediction[0])})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

**Explanation:**

- **joblib.load("iris_model.pkl")** → loads trained model.
- **/predict** endpoint accepts POST requests with JSON body.
- Example input: **{"features": [5.1, 3.5, 1.4, 0.2]}**.
- Returns: **{"prediction": 0}**.

### Build image

```bash
docker build -t iris-serving:latest .

## 4. Testing Containers Locally

### Training container:

```bash
    docker run iris-training:latest

- Trains and saves **iris_model.pkl**

### Serving container:

```bash
    docker run -p 5000:5000 iris-serving:latest

- Maps container port **5000** → local port **5000**

### Test API with curl:

```bash
    curl -X POST http://localhost:5000/predict \
         -H "Content-Type: application/json" \
         -d '{"features": [5.1, 3.5, 1.4, 0.2]}'

**Explanation:**

- **-X POST** → Send POST request.
- **-H "Content-Type: application/json"** → Tell API body is JSON.
- **-d '{"features": [...]}'** → Send sample features to predict.

### Expected output:

```json
{"prediction": 0}

## Key Insight

- Training and serving workflows should be separated into different containers.
- This ensures flexibility: train once, deploy many times.
- Containers make ML workflows reproducible and portable across environments.